# Bellameta Demo

In the daily work in digital pathology it is crucial to keep track of huge amounts of whole slide image (WSI) scans.
For that we offer micro services facilitating the integration
and metadata management of those scans.\
It is common practice that clinics encode valuable metadata in the filename of a scan. We need to extract this metadata and integrate it into a database.
This is the foundation for a consistent API allowing us to fetch this metadata for downstream tasks.

The purpose of this demo is to add metadata to a mocked example cohort of WSI scans within the storage database created by the service [bellastore](https://github.com/spang-lab/bellastore). This package takes off where `bellastore` ended. We asssume that we already have a database holding storage and ingress information of several WSI scans.

The philosophy of this package is to make the metadata integration as consistent across cohorts as possible and as automated as possible.\
Thus, the integration of a new cohort boils down to implementing a child class inheriting from an abstract base class which fills getter methods for the desired metadata to be included into the database.

We now explain step by step how this is achieved.💡

## The storage database

Under [data/scans.sqlite](../data/scans.sqlite) you find an exemplary storage database as generated by the `bellastore` package.
`bellameta` only operates on this database (and not on the actual scans in the filesystem).\
We also provide the [script](data/mock_sqlite.py) creating this mock database.

Looking at the storage table we see that that the scan names follow the layout `PatientId_Diagnosis_Year`.\
Looking at the ingress table we see that all scans were originally stored in a directory called `example_cohort`, i.e., all scans belong to this one cohort which is already important metadata.\
Furthermore, for this example, we assume that all slides are HE stained.

So it is time to add the metadata to the database.🤝
 

## Configuration

As metadata holds sensitive clinical information, the implementation of `bellameta` allows for use case specific configuration.

In your environment (in your cwd or higher directories) there must be a `.env` file defining the `BELLAMETA_CONFIG_PATH` variable, see also the [docs/.env](docs/.env). Under the path defined in `BELLAMETA_CONFIG_PATH` there has to be a `.yaml` file holding dictionaries
for specifying cohorts and tasks, see [bellameta.yaml](bellameta.yaml) for a minimal working example. These dictionaries are used to provide consistent typing, see [Typing of metadata](demo.ipynb#typing-of-metadata).

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Initializing metadata tables

First we need to initialize the desired metadata tables within the database. Therefore we stick to the default tables suggested by `bellameta`.

In [2]:
from bellameta.database import Db
from bellameta import types as t
from bellameta.base_metadata import Metadata

In [3]:
# Connecting to the database (located under DB_PATH)
db = Db()

Already existing metdata tables: ['state', 'cohort', 'patient', 'section', 'tag', 'stain', 'task', 'subtype', 'year', 'gleason_grade']


In [4]:
# Inspecting the metadata tables
print(str(db))

Table state:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table cohort:
                                            hash    value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY=  Example
1  sD0i_FFgiKliwOEkZ6uHKBSwq3dWa1SC2iWe1Sw75CM=  Example
2  d0OqHIuiHbL1mhI5W7thexq0Qrv4MBq1GQzZ4EwWNq4=  Example
3  Xt4KP7gVA-k_T3jAA7hCV1gTsl-9vqyz1Z64CqUT_RU=  Example
4  38_tlwKH0e2-7Gu3qG360I7J5EYoTc3KXpPjOXRb0KQ=  Example
Table patient:
                                            hash value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY=  7999
1  sD0i_FFgiKliwOEkZ6uHKBSwq3dWa1SC2iWe1Sw75CM=  9519
2  d0OqHIuiHbL1mhI5W7thexq0Qrv4MBq1GQzZ4EwWNq4=  9255
3  Xt4KP7gVA-k_T3jAA7hCV1gTsl-9vqyz1Z64CqUT_RU=  8987
4  38_tlwKH0e2-7Gu3qG360I7J5EYoTc3KXpPjOXRb0KQ=  7829
Table section:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table tag:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table stain:
                                            hash value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY

So we see that the default tables are now initialized as empty tables.
It is important to note, that all metadata tables follow the key (hash) value layout in order to relate to the storage and ingress tables.

In [5]:
# Inspecting a single table
print(db.read_pd('ingress').to_string())

                                           hash                                              filepath              filename
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY=    /tmp/tmpe52f2_do/example_cohort/7999_MCL_2015.ndpi    7999_MCL_2015.ndpi
1  sD0i_FFgiKliwOEkZ6uHKBSwq3dWa1SC2iWe1Sw75CM=  /tmp/tmpe52f2_do/example_cohort/9519_DLBCL_2020.ndpi  9519_DLBCL_2020.ndpi
2  d0OqHIuiHbL1mhI5W7thexq0Qrv4MBq1GQzZ4EwWNq4=  /tmp/tmpe52f2_do/example_cohort/9255_DLBCL_2015.ndpi  9255_DLBCL_2015.ndpi
3  Xt4KP7gVA-k_T3jAA7hCV1gTsl-9vqyz1Z64CqUT_RU=    /tmp/tmpe52f2_do/example_cohort/8987_CLL_2020.ndpi    8987_CLL_2020.ndpi
4  38_tlwKH0e2-7Gu3qG360I7J5EYoTc3KXpPjOXRb0KQ=     /tmp/tmpe52f2_do/example_cohort/7829_FL_2020.ndpi     7829_FL_2020.ndpi


Now we also directly see that all scans of our example cohort came from the ingress directory `/tmp/tmpe52f2_do/example_cohort`.

## Typing of metadata

The types module implements custom types for metdata. We have already implemented the `Cohort`, `Task`, `Subtype` and `Stain` type (using the Diagnosis and Stain type of [pamly-lib](https://github.com/spang-lab/pamly-lib)), but the modular structure allows to easily extend the typing.

Each type is a child class inheriting from `BellametaType` that receives a dictionary of admissible values for that type.

In order to not expose sensitive clinical data, these dicitonaries are accessed from a separate `.yaml` file whose path must be set in the `.env` file by the user, see [Configuration](demo.ipynb#configuration).
For this demo the `Cohort` and `Task` type use the [docs/bellameta.yaml](docs/bellameta.yaml) file for configuration.\
Users can simply extend the dictionaries in the `.yaml` file to automatically include more cohorts and tasks without any modifications to the code base.

`BellametaType` automatically takes over the implementation burden by dynamically setting up generic class methods:

In [8]:
# List registered cohorts
print(t.Cohort.list())

# Specifiy child class via string
print(t.Cohort.Example == t.Cohort('Example'))

# The typing allows to convert from int to class to string and vice verca
print(t.Cohort.from_int(0).to_string() == t.Cohort.Example.to_string())


['Example']
True
True


With this typing scheme we follow the `Diagnosis` and `Stain` types implemented in [pamly-lib](https://github.com/spang-lab/pamly-lib) accessible via the [pamly python package](https://pypi.org/project/pamly/).

## From filename to metadata

We now need to implement a cohort specific class that implements the logic of translating scan filenames into valid metadata.

For this we keep the implementation burden as low as possible by already providing the abstract base class `Metadata`. This class automates the communication with the database, serving as a simple API.

More precisely, each **cohort specific child class** only needs to implement a dedicated getter method for each metadata table that the user wishes to fill.

So let us implement such a child class for our example cohort. 🙌

*Note: If you use a sqlite database different to the one provided with the package you need to replace the absolute path specified in the following `__init__` function accordingly*

In [9]:
class Example(Metadata):
    '''
    An example class inheriting from Metadata.

    We imagine this class to implement the metadata of a new cohort of scans that just arrived from the clinic.
    '''

    def __init__(self, db: Db):
        # here we need to specify the absolute path to the ingress directory as given in the ingress table
        # this serves as the identifier for our cohort
        super().__init__(db, '/tmp/tmpe52f2_do/example_cohort')
    def get_cohort(self, hash):
        # typing for cohorts is provided via bellameta/types
        return t.Cohort.Example.to_string()
    def get_patient(self, hash):
        scanname = self.get_scanname_from_hash(hash)
        return scanname.split('_')[0]
    def get_year(self, hash):
        scanname = self.get_scanname_from_hash(hash)
        return scanname.split('_')[2]     
    def get_subtype(self, hash):
        scanname = self.get_scanname_from_hash(hash)
        diagnosis = scanname.split('_')[1]
        return t.Subtype.from_str(diagnosis).to_string()
    def get_stain(self, hash):
        return t.Stain.HE.to_string()
    def get_task(self, hash):
        # typing for tasks is provided via bellameta/types
        return [t.Task.Subtyping.to_string()]

So the `Example` class implements dedicated getter methods reading the desired metadata from the scan name or returning constant values.

## Writing metadata to database

In order to write the metadata to the database we just instantiate the `Example` class and then call the `write_many` method.

In [10]:
example = Example(db=db)
example.write_many()

Now let us check if everything is inserted properly into the respective metadata tables

In [11]:
print(str(db))

Table state:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table cohort:
                                            hash    value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY=  Example
1  sD0i_FFgiKliwOEkZ6uHKBSwq3dWa1SC2iWe1Sw75CM=  Example
2  d0OqHIuiHbL1mhI5W7thexq0Qrv4MBq1GQzZ4EwWNq4=  Example
3  Xt4KP7gVA-k_T3jAA7hCV1gTsl-9vqyz1Z64CqUT_RU=  Example
4  38_tlwKH0e2-7Gu3qG360I7J5EYoTc3KXpPjOXRb0KQ=  Example
Table patient:
                                            hash value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY=  7999
1  sD0i_FFgiKliwOEkZ6uHKBSwq3dWa1SC2iWe1Sw75CM=  9519
2  d0OqHIuiHbL1mhI5W7thexq0Qrv4MBq1GQzZ4EwWNq4=  9255
3  Xt4KP7gVA-k_T3jAA7hCV1gTsl-9vqyz1Z64CqUT_RU=  8987
4  38_tlwKH0e2-7Gu3qG360I7J5EYoTc3KXpPjOXRb0KQ=  7829
Table section:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table tag:
 Empty DataFrame
Columns: [hash, value]
Index: []
Table stain:
                                            hash value
0  AB7aew8E3oWkFRFHyf2aNzIwusWqPKzqmY9y18zeavY

So we see that all the metadata tables for which we have implemented getter methods are filled. Those, for which no getter method has been implemented remain unchanged. 🙌